In [ ]:
# I hand typed this all after watching the video, no claude at all. Come at me.

# ...but, I mean, I had to google a few things. My math chops are too rusty to
# do the derivative of tanh on the spot. Whatever, I'm still smart.

# Also the video used tanh everywhere, as far as I could tell, which means its
# incapable of outputs > 1 or < -1, and that's easy to fix, so I changed that.
# Besides that and some mostly arbitrary reorganizations of things, this should
# be pretty darn similar to the video.

# Also I hate notebooks, but I've decided to torture myself by getting used to
# them. So this is a single notebook instead of multiple files.

In [16]:
import math
import numpy as np
from graphviz import Digraph
import matplotlib.pyplot as plt
%matplotlib inline

In [66]:
class Value:
  def __init__(self, data, children=(), label="", op=""):
    self.data = data
    self._children = set(children)
    self._label = label
    self.grad = 0
    self._backward = lambda: None

    # only for graphviz/debugging, backward pass info captured at creation
    # via setting _backward in the operation that created this Value
    self._op = op


  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, children=(self, other), op="+")

    def _backward():
      self.grad += out.grad
      other.grad += out.grad
    
    out._backward = _backward
    return out
        
  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, children=(self, other), op="*")
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    
    out._backward = _backward
    return out

  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only floats and ints are supported"
    out = Value(self.data ** other, children=(self,), op=f"**{other}")

    def _backward():
      self.grad += other * self.data**(other-1) * out.grad

    out._backward = _backward
    return out
  
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, children=(self,), op="tanh")

    def _backward():
      self.grad += (1 - t**2) * out.grad
    
    out._backward = _backward
    return out

  def relu(self):
    out = Value(max(0, self.data), children=(self,), op=f"relu")

    def _backward():
      self.grad += (1 if self.data >= 0 else 0) * out.grad
    
    out._backward = _backward
    return out

  def _topo(self):
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._children:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    return topo

  def backward_from(self):
    nodes = self._topo()
    for n in reversed(nodes):
      n._backward()
    
  def _trace(self):
    nodes, edges = set(), set()
    def build(v):
      if v not in nodes:
        nodes.add(v)
        for child in v._children:
          edges.add((child, v))
          build(child)
    build(self)
    return nodes, edges
    
  def draw_from(self):
    dot = Digraph(format='svg', graph_attr={'rankdir':'LR'})

    nodes, edges = self._trace()
    for n in nodes:
      uid = str(id(n))
      dot.node(name = uid, label = f"{{{n._label} | {n.data:.4f} | grad: {n.grad:.2e}}}", shape='record')
      if n._op:
        dot.node(name = uid + n._op, label = n._op)
        dot.edge(uid + n._op, uid)

    for n1, n2 in edges:
      dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot
  
  def __neg__(self):
    return self * -1

  def __radd__(self, other):
    return self + other

  def __sub__(self, other):
    return self + (-other)

  def __rsub__(self, other):
    return other + (-self)
    
  def __rmul__(self, other):
    return self * other
  
  def __truediv__(self, other):
    return self * other**-1

  def __rtruediv__(self, other):
    return other / self
    
  def __repr__(self):
    return f"Value(data={self.data}, grad={self.grad})"

    

In [ ]:
class Neuron:
  def __init__(self, ins):
    